In [ ]:
# ============================================================
# CELL 1: GPU Check & Environment Setup
# ============================================================
import torch
import os

print("=" * 50)
print("GPU AVAILABILITY CHECK")
print("=" * 50)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   Total VRAM   : {total_mem:.2f} GB")
    DEVICE = torch.device("cuda")
else:
    print("⚠️  No GPU found — using CPU (training will be slow)")
    DEVICE = torch.device("cpu")

print(f"\nUsing device: {DEVICE}")

In [ ]:
# ============================================================
# CELL 2: Imports
# ============================================================
import os, random, shutil, warnings, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print("✅ All imports successful.")

In [ ]:
# ============================================================
# CELL 3: Configuration
# ============================================================

# ── Paths ──────────────────────────────────────────────────
DATASET_PATH   = Path("/kaggle/input/datasets/bapdesilva/betel-leaf-quality-dataset-2025/betel-quality-ds")
AUGMENTED_PATH = Path("/kaggle/working/augmented_dataset")
FINAL_PATH     = Path("/kaggle/working/final_dataset")

# ── Dataset ────────────────────────────────────────────────
TARGET_PER_CLASS = 2500
VAL_SPLIT        = 0.20
TEST_SPLIT       = 0.05 
IMG_SIZE         = 224

# ── Training hyper-parameters ──────────────────────────────
BATCH_SIZE   = 32
NUM_EPOCHS   = 30
LR_HEAD      = 1e-3
LR_FINE      = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 7
UNFREEZE_PCT = 0.30

CLASS_NAMES = [
    "Grade A Quality",
    "Grade B Quality",
    "Grade C Quality",
    "Grade D Quality",
    "Grade E Quality",
]
NUM_CLASSES = len(CLASS_NAMES)

print("✅ Configuration set.")
print(f"   Dataset path  : {DATASET_PATH}")
print(f"   Target/class  : {TARGET_PER_CLASS}")
print(f"   Val split     : {VAL_SPLIT*100:.0f}%")
print(f"   test split     : {TEST_SPLIT*100:.0f}%")
print(f"   Num classes   : {NUM_CLASSES}")
print(f"   Device        : {DEVICE}")